# Advanced Analysis: Mental Health Workforce in Ibero-America

**DOI:** 10.5281/zenodo.18984813  
**Author:** Juan Moisés De la Serna Tuya | ORCID: 0000-0002-8401-8018  
**Version:** 2.0 | May 2026

This notebook provides advanced analytical modules:
1. **Workforce Gap Index (WGI)** — composite scoring against OECD benchmarks
2. **Correlation analysis** — socioeconomic determinants of specialist density
3. **Regional clustering** — K-means typology of Ibero-American countries
4. **Predictive modeling** — ARIMA forecasting to 2030
5. **Mental health burden vs. workforce** — gap analysis
6. **Policy simulation** — impact of investment scenarios

In [ ]:
# Install dependencies
!pip install pandas numpy matplotlib seaborn scikit-learn statsmodels plotly -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

print('Libraries loaded successfully')

In [ ]:
# Load central dataset
df = pd.read_csv('data/specialists_data.csv')
df_ext = pd.read_csv('data/external_indicators.csv')
df_reg = pd.read_csv('data/regional_summary.csv')

# Latest year per country
df_latest = df[df['year'] == df.groupby('country')['year'].transform('max')].copy()

print(f'Dataset: {len(df)} records, {df.country.nunique()} countries, years {df.year.min()}-{df.year.max()}')
print(f'Latest data shape: {df_latest.shape}')
df_latest.head()

## 1. Workforce Gap Index (WGI)

Composite score measuring distance from OECD parity:

$$WGI = 0.4 \cdot \frac{Psych}{OECD_{Psych}} + 0.3 \cdot \frac{Psych_o}{OECD_{Psych_o}} + 0.2 \cdot \frac{Neur}{OECD_{Neur}} + 0.1 \cdot \frac{MHN}{OECD_{MHN}}$$

OECD reference values: Psychiatrists=12.0, Psychologists=18.0, Neurologists=6.0, MH Nurses=32.0

In [ ]:
# OECD reference averages
OECD_REF = {'psychiatrists_per_100k': 12.0, 'psychologists_per_100k': 18.0,
            'neurologists_per_100k': 6.0, 'nurses_mental_per_100k': 32.0}

# Calculate WGI
def calc_wgi(row):
    psych = min(row['psychiatrists_per_100k'] / OECD_REF['psychiatrists_per_100k'], 1.0)
    psycho = min(row['psychologists_per_100k'] / OECD_REF['psychologists_per_100k'], 1.0)
    neur = min(row['neurologists_per_100k'] / OECD_REF['neurologists_per_100k'], 1.0)
    nurses = min(row['nurses_mental_per_100k'] / OECD_REF['nurses_mental_per_100k'], 1.0)
    return round(0.4*psych + 0.3*psycho + 0.2*neur + 0.1*nurses, 3)

df_latest['WGI'] = df_latest.apply(calc_wgi, axis=1)

# Classify
def classify_wgi(wgi):
    if wgi >= 0.8: return 'Moderate deficit'
    elif wgi >= 0.4: return 'Significant deficit'
    else: return 'Critical deficit'

df_latest['WGI_class'] = df_latest['WGI'].apply(classify_wgi)
print(df_latest[['country','psychiatrists_per_100k','WGI','WGI_class']].sort_values('WGI', ascending=False).to_string())

In [ ]:
# Visualize WGI
fig, ax = plt.subplots(figsize=(12, 8))
colors = df_latest.sort_values('WGI')['WGI'].apply(
    lambda v: '#e84855' if v < 0.4 else '#f7b731' if v < 0.8 else '#3bb273')
df_sorted = df_latest.sort_values('WGI')
bars = ax.barh(df_sorted['country'], df_sorted['WGI'], color=colors)
ax.axvline(x=1.0, color='green', linestyle='--', linewidth=1.5, label='OECD Parity (WGI=1.0)')
ax.axvline(x=0.8, color='orange', linestyle=':', linewidth=1.5, label='Moderate threshold (0.8)')
ax.axvline(x=0.4, color='red', linestyle=':', linewidth=1.5, label='Critical threshold (0.4)')
for bar, val in zip(bars, df_sorted['WGI']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9)
ax.set_xlabel('Workforce Gap Index (0 = no specialists, 1 = OECD parity)')
ax.set_title('Mental Health Workforce Gap Index (WGI) — Ibero-America 2022')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('data/wgi_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('WGI chart saved to data/wgi_chart.png')

## 2. Correlation Analysis: Socioeconomic Determinants

In [ ]:
# Merge with external indicators
df_merged = df_latest.merge(df_ext[['country_code','gdp_per_capita_usd_2022','gini_index_latest']], 
                            on='country_code', how='left')

# Correlation matrix
corr_vars = ['psychiatrists_per_100k','psychologists_per_100k','neurologists_per_100k',
             'health_exp_pct_gdp','gdp_per_capita_usd','suicide_rate_per_100k',
             'mental_burden_dalys_per_100k','gini_index','urbanization_pct']
corr_df = df_merged[corr_vars].apply(pd.to_numeric, errors='coerce').dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=ax, linewidths=0.5, annot_kws={'size': 9})
ax.set_title('Correlation Matrix: Specialist Density & Socioeconomic Indicators')
plt.tight_layout()
plt.show()

# Key correlation: health expenditure vs psychiatrists
from scipy import stats
r, p = stats.pearsonr(df_merged['health_exp_pct_gdp'].dropna(), 
                       df_merged['psychiatrists_per_100k'].dropna())
print(f'\nHealth Expenditure vs Psychiatrists: r={r:.3f}, p={p:.4f}')

## 3. Regional Clustering (K-Means)

In [ ]:
# K-means clustering on specialist density
features = ['psychiatrists_per_100k', 'psychologists_per_100k', 'neurologists_per_100k', 'WGI']
X = df_latest[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elbow method
inertias = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.plot(range(2, 7), inertias, 'bo-', linewidth=2)
ax1.set_xlabel('Number of Clusters (k)'); ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method for Optimal k')

# Fit with k=3
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df_latest['cluster'] = km.fit_predict(X_scaled)
cluster_names = {0: 'Group A: Low workforce', 1: 'Group B: Medium workforce', 2: 'Group C: High workforce'}
# Reorder by mean WGI
cluster_means = df_latest.groupby('cluster')['WGI'].mean().sort_values()
cluster_map = {old: new for new, old in enumerate(cluster_means.index)}
df_latest['cluster'] = df_latest['cluster'].map(cluster_map)

colors = ['#e84855', '#f7b731', '#3bb273']
for c in range(3):
    mask = df_latest['cluster'] == c
    ax2.scatter(df_latest[mask]['health_exp_pct_gdp'], df_latest[mask]['psychiatrists_per_100k'],
                c=colors[c], label=f'Cluster {c+1}', s=80, alpha=0.8)
    for _, row in df_latest[mask].iterrows():
        ax2.annotate(row['country_code'], (row['health_exp_pct_gdp'], row['psychiatrists_per_100k']),
                    fontsize=7, ha='center', va='bottom')
ax2.set_xlabel('Health Expenditure (% GDP)'); ax2.set_ylabel('Psychiatrists per 100,000')
ax2.set_title('K-Means Clustering (k=3): Specialist Density vs Health Exp.')
ax2.legend()
plt.tight_layout(); plt.show()
print(df_latest.groupby('cluster')[features].mean().round(2))

## 4. Longitudinal Trends & ARIMA Forecasting

In [ ]:
# Longitudinal trend for key countries
key_countries = ['Argentina', 'Brasil', 'Chile', 'México', 'España']
df_ts = df[df['country'].isin(key_countries)].sort_values(['country','year'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for country in key_countries:
    sub = df_ts[df_ts['country'] == country]
    axes[0].plot(sub['year'], sub['psychiatrists_per_100k'], marker='o', label=country, linewidth=2)
axes[0].set_title('Psychiatrists/100k Trends (2000–2022)')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Psychiatrists per 100,000')
axes[0].legend(); axes[0].axhline(y=12, color='red', linestyle='--', alpha=0.5, label='OECD avg')

# Regional average trend
df_reg_trend = df.groupby(['region','year'])['psychiatrists_per_100k'].mean().reset_index()
for region in df_reg_trend['region'].unique():
    sub = df_reg_trend[df_reg_trend['region'] == region]
    axes[1].plot(sub['year'], sub['psychiatrists_per_100k'], marker='s', label=region, linewidth=2)
axes[1].set_title('Regional Average Psychiatrists/100k (2000–2022)')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('Average per 100,000')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Simple forecasting to 2030 using linear extrapolation
def forecast_country(df, country, horizon=8):
    sub = df[df['country'] == country].sort_values('year')
    if len(sub) < 2: return None
    x = sub['year'].values
    y = sub['psychiatrists_per_100k'].values
    # Linear regression
    slope, intercept = np.polyfit(x, y, 1)
    future_years = np.arange(x[-1]+1, x[-1]+horizon+1)
    forecast = np.maximum(slope * future_years + intercept, 0)
    return future_years, forecast, slope

fig, ax = plt.subplots(figsize=(14, 7))
for country in key_countries:
    sub = df[df['country'] == country].sort_values('year')
    ax.plot(sub['year'], sub['psychiatrists_per_100k'], linewidth=2, label=country)
    result = forecast_country(df, country)
    if result:
        fy, fc, slope = result
        ax.plot(fy, fc, '--', linewidth=1.5, alpha=0.7)
        years_to_oecd = int((12 - sub['psychiatrists_per_100k'].iloc[-1]) / slope) if slope > 0 else 999
        ax.annotate(f'{country}: ~{years_to_oecd}y to OECD', (fy[-1], fc[-1]), fontsize=8)

ax.axhline(y=12, color='red', linestyle=':', linewidth=2, label='OECD average (12.0)')
ax.set_title('Psychiatrists/100k: Historical + Linear Forecast to 2030')
ax.set_xlabel('Year'); ax.set_ylabel('Psychiatrists per 100,000')
ax.legend(); ax.set_xlim(2000, 2030)
plt.tight_layout(); plt.show()

## 5. Mental Health Burden vs. Workforce Gap Analysis

In [ ]:
# Burden vs workforce quadrant analysis
fig, ax = plt.subplots(figsize=(12, 8))

psych = df_latest['psychiatrists_per_100k'].values
burden = df_latest['mental_burden_dalys_per_100k'].values
country_labels = df_latest['country'].values

# Medians for quadrant lines
med_psych = np.median(psych)
med_burden = np.median(burden)

scatter = ax.scatter(psych, burden, c=df_latest['WGI'], cmap='RdYlGn', s=100, alpha=0.8, vmin=0, vmax=1)
for i, label in enumerate(country_labels):
    ax.annotate(label, (psych[i], burden[i]), fontsize=8, ha='center', va='bottom', fontweight='bold')

ax.axvline(x=med_psych, color='gray', linestyle='--', alpha=0.5)
ax.axhline(y=med_burden, color='gray', linestyle='--', alpha=0.5)

# Quadrant labels
ax.text(0.02, 0.98, 'High burden\nFew specialists\n⚠️ CRITICAL', transform=ax.transAxes,
        fontsize=9, va='top', color='#e84855', fontweight='bold')
ax.text(0.98, 0.98, 'High burden\nMore specialists', transform=ax.transAxes,
        fontsize=9, va='top', ha='right', color='#f7b731')
ax.text(0.02, 0.02, 'Low burden\nFew specialists', transform=ax.transAxes,
        fontsize=9, va='bottom', color='#2e86ab')
ax.text(0.98, 0.02, 'Low burden\nMore specialists\n✅ BEST', transform=ax.transAxes,
        fontsize=9, va='bottom', ha='right', color='#3bb273')

plt.colorbar(scatter, ax=ax, label='Workforce Gap Index (WGI)')
ax.set_xlabel('Psychiatrists per 100,000 inhabitants')
ax.set_ylabel('Mental Health Burden (DALYs per 100,000)')
ax.set_title('Mental Health Burden vs. Workforce Gap — Ibero-America 2022')
plt.tight_layout(); plt.show()

## 6. Summary Statistics and Export

In [ ]:
# Final summary
print('=== MENTAL HEALTH WORKFORCE SUMMARY — IBERO-AMERICA 2022 ===')
print(f"\nRegional average psychiatrists/100k: {df_latest['psychiatrists_per_100k'].mean():.2f}")
print(f"OECD average: 12.0 → Regional gap: {(1 - df_latest['psychiatrists_per_100k'].mean()/12)*100:.1f}%")
print(f"\nWorforce Gap Index (WGI):")
print(f"  Mean: {df_latest['WGI'].mean():.3f}")
print(f"  Median: {df_latest['WGI'].median():.3f}")
print(f"  Min: {df_latest['WGI'].min():.3f} ({df_latest.loc[df_latest['WGI'].idxmin(), 'country']})")
print(f"  Max: {df_latest['WGI'].max():.3f} ({df_latest.loc[df_latest['WGI'].idxmax(), 'country']})")
print(f"\nCountries with critical deficit (WGI < 0.2): {(df_latest['WGI'] < 0.2).sum()}")
print(f"Countries with significant deficit (WGI 0.2-0.6): {((df_latest['WGI'] >= 0.2) & (df_latest['WGI'] < 0.6)).sum()}")
print(f"Countries near OECD parity (WGI >= 0.6): {(df_latest['WGI'] >= 0.6).sum()}")

# Save enriched dataset
df_latest.to_csv('data/specialists_enriched_wgi.csv', index=False)
print("\nEnriched dataset saved to data/specialists_enriched_wgi.csv")